## **Reflections (week 5 - 8)**   
## **Name : Roozbeh Khodayari**

In [1]:
# import necessary libraries
import numpy as np
import pandas as pd
import statsmodels.api as sm

### **Homework Reflection 9**

**1. Write some code that will use a simulation to estimate the standard deviation of the coefficient when there is heteroskedasticity.**   
**Compare these standard errors to those found via statsmodels OLS or a similar linear regression model.**


In [ ]:
rng = np.random.default_rng(42)
# simulate data with heteroskedasticity
def simulate_hetero(A=1, B=1, C=10, D=1000):
    W = rng.normal(0, 1, D)
    X = W + rng.normal(0, B, D)

    # Error standard deviation increases with X
    noise_sd = C * np.exp(0.4 * X)
    Y = A * X - W + rng.normal(0, noise_sd, D)

    return Y, X, W

# calculate standard errors 
n_sim = 2000
coefficients = []
ols_ses = []
robust_ses = []

for _ in range(n_sim):
    Y, X, W = simulate_hetero()

    X_reg = sm.add_constant(np.column_stack((X, W)))
    model = sm.OLS(Y, X_reg).fit()

    coefficients.append(model.params[1])
    ols_ses.append(model.bse[1])
    robust_ses.append(model.get_robustcov_results(cov_type="HC3").bse[1])

print(f"Simulation SD:  {np.std(coefficients, ddof=1):.4f}")
print(f"Mean OLS SE:    {np.mean(ols_ses):.4f}")
print(f"Mean HC3 SE:    {np.mean(robust_ses):.4f}")
print(f"Mean X coef:    {np.mean(coefficients):.4f}")


Simulation SD:  0.5470
Mean OLS SE:    0.4357
Mean HC3 SE:    0.5548
Mean X coef:    0.9923


I simulated data with heteroskedasticity by making the error standard deviation depend on X. I ran 2,000 simulations and calculated the standard deviation of the estimated X coefficients.

The simulated standard deviation was 0.547, while the average standard error from regular OLS was 0.436. Regular OLS standard errors assume constant error variance (homoskedasticity), which was not true in my simulation. I also used HC3 robust standard errors, which do not require constant error variance. The average HC3 standard error was 0.555, much closer to the simulated result.

This helped me understand that OLS coefficients can still be unbiased under heteroskedasticity, but their conventional standard errors may be inaccurate. In this case, HC3 provided a more reliable estimate of uncertainty.

**2. Write some code that will use a simulation to estimate the standard deviation of the coefficient when errors are highly correlated / non-independent.**  
**Compare these standard errors to those found via statsmodels OlS or a similar linear regression model.**   

**Show that if the correlation between errors is high enough, then the estimated standard deviation of the coefficient, using bootstrap errors,**   
**might not match that found by a full simulation of the Data Generating Process.  (This can be fixed if you have a huge amount of data for the bootstrap simulation.)**

In [3]:
rng = np.random.default_rng(42)
n_groups, group_size = 30, 20
n = n_groups * group_size
groups = np.repeat(np.arange(n_groups), group_size)

def simulate(n_groups=30, group_size=20):
    # Group-level X creates similar X values within the same group.
    group_x = rng.normal(0, 1, n_groups)
    x = group_x[groups] + rng.normal(0, 0.3, n)

    # The group-shared error produces high within-group correlation.
    group_error = rng.normal(0, 3, n_groups)
    error = group_error[groups] + rng.normal(0, 0.6, n)
    y = 1 + x + error  # True coefficient of X is 1.
    return x, y

In [4]:
# Full data-generating-process simulation, plus conventional OLS SE.
coefficients = []
ols_ses = []

for _ in range(2000):
    x, y = simulate()
    model = sm.OLS(y, sm.add_constant(x)).fit()
    coefficients.append(model.params[1])
    ols_ses.append(model.bse[1])

simulation_sd = np.std(coefficients, ddof=1)
mean_ols_se = np.mean(ols_ses)


In [5]:
# Ordinary row bootstrap: resample observations from ONE dataset.
x, y = simulate()
bootstrap_coefficients = []

for _ in range(1500):
    idx = rng.integers(0, n, size=n)
    model = sm.OLS(y[idx], sm.add_constant(x[idx])).fit()
    bootstrap_coefficients.append(model.params[1])

bootstrap_sd = np.std(bootstrap_coefficients, ddof=1)
print(f"Full simulation coefficient SD: {simulation_sd:.4f}")
print(f"Mean conventional OLS SE:      {mean_ols_se:.4f}")
print(f"Ordinary row bootstrap SD:     {bootstrap_sd:.4f}")
print(f"Mean estimated X coefficient:  {np.mean(coefficients):.4f}")


Full simulation coefficient SD: 0.5139
Mean conventional OLS SE:      0.1199
Ordinary row bootstrap SD:     0.1467
Mean estimated X coefficient:  0.9924


I generated data with strongly correlated errors within groups and ran 2,000 simulations. The standard deviation of the estimated X coefficients was 0.514, while the average conventional OLS standard error was only 0.120. The conventional OLS standard error underestimated uncertainty because it assumed independent errors.

I then used ordinary bootstrap resampling on one dataset. The bootstrap standard deviation was 0.147, still much smaller than the full simulation result. Resampling individual rows did not properly account for the correlation among observations in the same group.

This showed me that ordinary bootstrap can give misleading standard errors when errors are highly correlated. Having more independent groups and using a bootstrap method that preserves the group structure could improve the results. However, simply increasing the number of bootstrap repetitions would not fix the problem.

### **Homework Reflection 10**

We are skipping this one, this time.

### **Homework Reflection 11**

1.Construct a dataset for an event study where the value, derivative, and second derivative of a trend all change discontinuously (suddenly) after an event.
Build a model that tries to decide whether the event is real (has a nonzero effect) using:
(a) only the value,
(b) the value, derivative, and second derivative.
Which of these models is better at detecting and/or quantifying the impact of the event?  (What might "better" mean here?)

2.Construct a dataset in which there are three groups whose values each increase discontinuously (suddenly) by the same amount at a shared event; they change in parallel
over time, but they have different starting values.  Create a model that combines group fixed effects with an event study, as suggested in the online reading.
Explain what you did, how the model works, and how it accounts for both baseline differences and the common event effect.

### **Homework Reflection 12**

Construct a dataset in which prior trends do not hold, and in which this makes the differences-in-differences come out wrong.  Explain why the
differences-in-differences estimate of the effect comes out higher or lower than the actual effect.
